[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/07_modern_architectures/07_modern_architectures.ipynb)

# 07 · 现代架构：RoPE / GQA / MoE / SSM

<span style="background:#1a7f37;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 PyTorch，无需 GPU、无需下载模型权重（最后一节只在线拉取一个几 KB 的 config.json，并带离线回退）。

本课收尾模块。前六个模块搭好了一个 GPT-2 风格 decoder 并榨干了它的训练与推理；本 notebook 亲手实现把它升级为 Llama/Qwen/Mixtral/Mamba 的四个关键组件：

| 节 | 实现什么 | 对应讲义 |
|---|---|---|
| 1 | **RoPE**：`precompute_freqs` + `apply_rope`，并数值验证"内积只依赖相对位置" | §2 [Su 2021] |
| 2 | **GQA**：`repeat_kv` 头共享 + MHA/GQA/MQA 参数量与 KV cache 对照表 | §3 [Shazeer 2019; Ainslie 2023] |
| 3 | **Toy MoE**：4 专家 + top-2 router + 负载均衡损失，演示 router 坍缩与修复 | §5 [Shazeer 2017; Fedus 2021] |
| 4 | **SSM 玩具**：线性递推 scan，与 attention 在"精确召回"任务上的行为分界 | §6 [Gu & Dao 2023] |
| 5 | **真实 config 对照**：Qwen2.5-0.5B 的 config 逐字段指认本模块概念 | §1–§4 |

然后是 3 道 ✏️ 练习 + 📖 参考答案 + 全课总结。

In [ ]:
import math
import torch

torch.manual_seed(7)

# ============ 1. RoPE：旋转位置编码 [Su 2021] ============
# 把 d 维向量拆成 d/2 个二维子空间，位置 m 处第 i 对维度旋转角度 m*theta_i，
# theta_i = theta^(-2i/d)：i 小转得快（分辨相邻 token），i 大转得慢（编码长程）。

def precompute_freqs(d_head, max_len, theta=10000.0):
    # 返回 cos/sin 表，形状各为 (max_len, d_head/2)
    inv_freq = theta ** (-torch.arange(0, d_head, 2).float() / d_head)   # (d/2,)  = theta^(-2i/d)
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)        # (max_len, d/2)，第 m 行 = m*theta_i
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    # x: (..., T, d)。维度按 (0,1),(2,3),... 配对，第 m 行（位置 m）做二维旋转
    T = x.shape[-2]
    c, s = cos[:T], sin[:T]                       # (T, d/2)，广播到 batch/head 维
    x1, x2 = x[..., 0::2], x[..., 1::2]           # 每对的两个分量，各 (..., T, d/2)
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s              # 旋转矩阵 [[c,-s],[s,c]] 逐对作用
    out[..., 1::2] = x1 * s + x2 * c
    return out

# --- 体检：旋转是正交变换 → 保范数；位置 0 角度为 0 → 恒等 ---
cos_t, sin_t = precompute_freqs(d_head=64, max_len=512)
q = torch.randn(1, 8, 512, 64)                    # (B, n_head, T, d_head)
q_rot = apply_rope(q, cos_t, sin_t)
print("范数保持   :", torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-5))
print("位置 0 恒等:", torch.allclose(q[..., 0, :], q_rot[..., 0, :], atol=1e-6))

# ============ 数值验证关键性质：<R_m q, R_n k> 只依赖 m-n ============
# 构造两组绝对位置完全不同、相对距离相同的 (q,k)，attention 分数应完全相等。
qv, kv_vec = torch.randn(64), torch.randn(64)

def rope_at(v, m):
    # 把单个向量 v 放到绝对位置 m（嵌入零序列第 m 行，旋转后取回）
    z = torch.zeros(512, 64)
    z[m] = v
    return apply_rope(z, cos_t, sin_t)[m]

s_a = rope_at(qv, 3)   @ rope_at(kv_vec, 7)      # 相对距离 4
s_b = rope_at(qv, 103) @ rope_at(kv_vec, 107)    # 相对距离也是 4，绝对位置 +100
s_c = rope_at(qv, 3)   @ rope_at(kv_vec, 30)     # 相对距离 27（对照组）
print(f"score(m=3,  n=7)   = {s_a.item():+.6f}")
print(f"score(m=103,n=107) = {s_b.item():+.6f}   <- 与上行相等：绝对位置无关")
print(f"score(m=3,  n=30)  = {s_c.item():+.6f}   <- 相对距离不同 → 分数不同")
assert torch.allclose(s_a, s_b, atol=1e-4)
assert not torch.allclose(s_a, s_c, atol=1e-2)
print("✅ 验证通过：RoPE 下 attention 分数只依赖相对位置 m-n")

In [ ]:
# ============ 2. GQA：分组共享 KV 头 [Shazeer 2019 (MQA); Ainslie 2023 (GQA)] ============
# n_head 个 query 头分成 n_kv 组，组内共享同一组 K/V → KV cache 直接缩小 n_head/n_kv 倍。

def repeat_kv(kv, n_rep):
    # kv: (B, n_kv, T, d_head) -> (B, n_kv*n_rep, T, d_head)
    # 把每个 KV 头复制 n_rep 份，供同组的 n_rep 个 query 头使用（计算等价、实现最简的写法）
    if n_rep == 1:
        return kv
    B, n_kv, T, dh = kv.shape
    return kv[:, :, None, :, :].expand(B, n_kv, n_rep, T, dh).reshape(B, n_kv * n_rep, T, dh)

def gqa_attention(q, k, v):
    # q: (B, n_head, T, dh)；k/v: (B, n_kv, T, dh)
    n_rep = q.shape[1] // k.shape[1]
    k, v = repeat_kv(k, n_rep), repeat_kv(v, n_rep)
    att = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    mask = torch.triu(torch.ones(q.shape[2], q.shape[2], dtype=torch.bool), diagonal=1)
    att = att.masked_fill(mask, float("-inf"))
    return torch.softmax(att, dim=-1) @ v

B, n_head, n_kv, T, dh = 2, 8, 2, 16, 32
out = gqa_attention(torch.randn(B, n_head, T, dh),
                    torch.randn(B, n_kv, T, dh),
                    torch.randn(B, n_kv, T, dh))
print("GQA 前向输出形状:", tuple(out.shape), " (8 个 query 头共享 2 个 KV 头)\n")

# ============ 记账：MHA / GQA / MQA 对照（复用模块 06 的记账思想）============
def attn_accounting(d_model, n_head, n_kv, n_layer, seq_len, bytes_per=2):
    # 参数：W_q (d,d) + W_o (d,d) + W_k/W_v 各 (d, n_kv*dh)，现代配方无 bias
    dh = d_model // n_head
    params = (2 * d_model * d_model + 2 * d_model * n_kv * dh) * n_layer
    # KV cache：2(K和V) × L × n_kv × dh × T × 字节数（fp16=2）
    kv_cache = 2 * n_layer * n_kv * dh * seq_len * bytes_per
    return params, kv_cache

# Llama-2-70B 的真实注意力形状：d=8192，64 个 query 头，L=80，4096 上下文，fp16
print(f"{'配置':<42}{'attn 参数':>12}{'KV cache/序列':>16}")
for name, nkv in [("MHA  n_kv=64（假设不用 GQA）", 64),
                  ("GQA  n_kv=8  ← Llama-2-70B 实际配置", 8),
                  ("MQA  n_kv=1", 1)]:
    p, c = attn_accounting(8192, 64, nkv, 80, 4096)
    print(f"{name:<42}{p/1e9:>10.2f} B{c/2**30:>12.2f} GiB")
print("\n→ GQA(8) 把 10 GiB/序列的 KV cache 压到 1.25 GiB：同一张卡的并发量直接 ×8。")
print("→ 质量上 GQA 经 ~5% 算力的 uptrain 后逼近 MHA [Ainslie 2023]——这笔交易没有理由不做。")

In [ ]:
# ============ 3. Toy MoE：router + top-2 门控 + 负载均衡损失 ============
# [Shazeer 2017] 稀疏门控；[Fedus 2021] Switch 的负载均衡辅助损失 L_aux = E * Σ_i f_i * P_i
#   f_i = 实际路由到专家 i 的 token 占比（硬计数，stop-grad）
#   P_i = router 给专家 i 的平均 softmax 概率（可导，梯度由此流回 router）
# 均匀分配时 f_i = P_i = 1/E，L_aux = 1 为最小值；越倾斜越大。

class ToyMoE(torch.nn.Module):
    def __init__(self, d, d_ff, n_experts=4, top_k=2):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.router = torch.nn.Linear(d, n_experts)        # 一个线性层就是全部的 router
        self.experts = torch.nn.ModuleList(
            torch.nn.Sequential(torch.nn.Linear(d, d_ff), torch.nn.SiLU(), torch.nn.Linear(d_ff, d))
            for _ in range(n_experts))

    def forward(self, x):                                   # x: (N, d) —— N 个 token
        probs = torch.softmax(self.router(x), dim=-1)       # (N, E)
        gate, idx = probs.topk(self.top_k, dim=-1)          # 各 (N, k)：top-k 门控
        gate = gate / gate.sum(-1, keepdim=True)            # 选中专家内重归一化
        y = torch.zeros_like(x)
        for e in range(self.n_experts):                     # 稀疏分发：每个专家只算分给它的 token
            for slot in range(self.top_k):
                m = idx[:, slot] == e
                if m.any():
                    y[m] = y[m] + gate[m, slot:slot+1] * self.experts[e](x[m])
        f = torch.stack([(idx == e).float().mean() for e in range(self.n_experts)])  # 占比（含 k 个槽位）
        P = probs.mean(dim=0)
        aux = self.n_experts * (f.detach() * P).sum()       # Switch 式 L_aux
        counts = torch.bincount(idx.flatten(), minlength=self.n_experts)
        return y, aux, counts

def show_routing(counts, title):
    total = counts.sum().item()
    print(title)
    for e, c in enumerate(counts.tolist()):
        bar = "█" * round(40 * c / total)
        print(f"  expert {e}: {bar:<40} {c/total:6.1%}")

torch.manual_seed(0)
moe = ToyMoE(d=32, d_ff=64, n_experts=4, top_k=2)
with torch.no_grad():                       # 人为制造"赢家通吃"初始化 → router 坍缩
    moe.router.bias[0] += 3.0               # 专家 0/1 的 logit 恒定占优 → top-2 几乎总选它们
    moe.router.bias[1] += 1.5
x_tokens = torch.randn(2048, 32)
_, aux, counts = moe(x_tokens)
show_routing(counts, f"坍缩态（专家 0/1 吃掉几乎所有 token）   L_aux = {aux.item():.3f}")

opt = torch.optim.Adam(moe.router.parameters(), lr=2e-2)
for step in range(300):                     # 只优化 L_aux，看 router 被推回均匀
    _, aux, _ = moe(torch.randn(512, 32))
    opt.zero_grad(); aux.backward(); opt.step()

_, aux, counts = moe(x_tokens)
show_routing(counts, f"\n加负载均衡损失训练 300 步后            L_aux = {aux.item():.3f}（完全均匀时 ≈ 1.0）")
print("\n→ 参数量 ∝ E（4 份专家 MLP），每 token FLOPs ∝ k（只算 2 份）：参数与计算解耦。")
print("→ 没有 L_aux 时坍缩是正反馈：分到 token 多的专家学得更好 → 分到更多。真实训练必须带它。")

In [ ]:
# ============ 4. SSM 玩具：线性递推 scan vs attention 的"精确召回" [Gu & Dao 2023] ============
# 最简对角 SSM：h_t = a ⊙ h_{t-1} + x_t（a∈(0,1) 是保留率；Mamba 的 selective 即让
# a、输入系数依赖于 x_t）。推理时只维护一个固定大小的 h —— O(1)/token、常数内存；
# 代价：全部历史被有损压缩进 h，早期 token 的系数按 a^Δt 指数稀释。

def ssm_scan(xs, a):
    # xs: (T, d) -> hs: (T, d)，沿时间线性递推
    h = torch.zeros_like(xs[0])
    hs = []
    for x_t in xs:
        h = a * h + x_t
        hs.append(h)
    return torch.stack(hs)

# 玩具召回任务：开头放一个目标 token，后接 T-1 个干扰 token，末尾问"开头是什么"。
# attention 用内容匹配 softmax 查表（无损随机访问）；SSM 只能从 h_T 里"挖"。
d, a = 64, 0.9
print(f"{'T':>6} | {'SSM: x_0 在 h_T 中的占比':>26} | {'attention: pos-0 的权重':>24}")
for T in [4, 16, 64, 256, 1024]:
    coef = a ** torch.arange(T - 1, -1, -1).float()        # x_t 在 h_T 中的系数 a^(T-1-t)
    ssm_share = (coef[0] / coef.sum()).item()              # 目标 token 的信息占比
    keys = torch.randn(T, d)
    keys = keys / keys.norm(dim=-1, keepdim=True)          # 单位 key
    q_match = 16.0 * keys[0]                               # query 与 key_0 强内容匹配
    attn_w0 = torch.softmax(keys @ q_match, dim=0)[0].item()
    print(f"{T:>6} | {ssm_share:>26.2e} | {attn_w0:>24.4f}")

# 行为分界的直接展示：从最终状态/输出里还原目标 token
vals = torch.randn(1024, d)
attn_out = (torch.softmax((vals / vals.norm(dim=-1, keepdim=True)) @ \
            (16.0 * vals[0] / vals[0].norm()), dim=0)[:, None] * vals).sum(0)
ssm_out = ssm_scan(vals, a)[-1]
print(f"\n与目标 x_0 的余弦相似度（T=1024）: attention 检索 = "
      f"{torch.cosine_similarity(attn_out, vals[0], dim=0).item():.3f}, "
      f"SSM 末状态 = {torch.cosine_similarity(ssm_out, vals[0], dim=0).item():.3f}")
print("→ attention 的检索不随距离衰减；SSM 的固定状态装不下任意长历史（信息论必然）。")
print("→ 这就是 SSM 在 needle-in-a-haystack / 精确复制评测上系统性吃亏的架构根源，")
print("  也是 hybrid（多数 Mamba 层 + 少量 attention 层）成为 2024 后主流配方的原因。")

In [ ]:
# ============ 5. 真实模型 config 对照：Qwen/Qwen2.5-0.5B ============
# 只拉取几 KB 的 config.json（不下载任何权重）；离线时回退到内嵌的真实字段。
FALLBACK_QWEN25_05B = {
    "hidden_size": 896, "num_hidden_layers": 24,
    "num_attention_heads": 14, "num_key_value_heads": 2,
    "intermediate_size": 4864, "hidden_act": "silu",
    "rope_theta": 1000000.0, "max_position_embeddings": 32768,
    "rms_norm_eps": 1e-06, "vocab_size": 151936, "tie_word_embeddings": True,
}
try:
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-0.5B").to_dict()
    print("已在线加载 Qwen/Qwen2.5-0.5B 的 config\n")
except Exception as e:
    cfg = dict(FALLBACK_QWEN25_05B)
    print(f"离线回退（{type(e).__name__}），使用内嵌 config\n")

notes = {
    "hidden_size":            "d_model = 896",
    "num_hidden_layers":      "L = 24",
    "num_attention_heads":    "14 个 query 头（d_head = 896/14 = 64）",
    "num_key_value_heads":    "2 个 KV 头 → GQA！KV cache 仅为 MHA 的 2/14 ≈ 1/7（本章 §3）",
    "intermediate_size":      "SwiGLU 的 d_ff = 4864 ≈ 5.4×d（Qwen 比 Llama 的 8/3 系数更宽）（§4）",
    "hidden_act":             "silu → 即 SwiGLU 门控 FFN（§4）",
    "rope_theta":             "RoPE 底数 b = 1e6（默认是 1e4）→ 为 32K 长上下文铺长波长（§2）",
    "max_position_embeddings": "训练上下文 32768",
    "rms_norm_eps":           "出现 rms_norm → 用的是 RMSNorm 而非 LayerNorm（§4）",
}
for k, note in notes.items():
    print(f"{k:<25} = {cfg.get(k)!s:<10}  # {note}")
print("\n→ 一个 0.5B 的小模型，把本模块的现代配方（RoPE/GQA/RMSNorm/SwiGLU）一个不落地用全了。")

## ✏️ 练习 1：实现 `apply_rope`

不看上面的实现，从公式独立写一遍 RoPE（这是现代 LLM 源码里出现频率最高的 50 行之一）。

**任务**：实现 `apply_rope_ex(x, cos, sin)`：
- `x` 形状 `(..., T, d)`；`cos`/`sin` 形状 `(max_len, d/2)`（由已给的 `precompute_freqs` 生成）；
- 维度按 `(0,1), (2,3), ...` 配对，位置 $m$ 处第 $i$ 对按角度 $m\theta_i$ 旋转：
  $(x_1, x_2) \mapsto (x_1\cos - x_2\sin,\; x_1\sin + x_2\cos)$。

**提示**：`x[..., 0::2]` / `x[..., 1::2]` 取出每对的两个分量；`cos[:T]` 会自动广播到 batch/head 维；约 8 行。

In [ ]:
def apply_rope_ex(x, cos, sin):
    # x: (..., T, d)；cos/sin: (max_len, d/2)
    # TODO: 1) 取出前 T 行的 cos/sin
    # TODO: 2) x1, x2 = 偶数下标维度, 奇数下标维度
    # TODO: 3) 按旋转公式写回 out 的偶/奇下标位置并返回
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
cos64, sin64 = precompute_freqs(64, 256)
x_test = torch.randn(2, 4, 128, 64)
y_test = apply_rope_ex(x_test, cos64, sin64)
assert y_test.shape == x_test.shape, "形状必须不变"
assert torch.allclose(y_test.norm(dim=-1), x_test.norm(dim=-1), atol=1e-5), "旋转必须保范数"
assert torch.allclose(y_test[..., 0, :], x_test[..., 0, :], atol=1e-6), "位置 0 角度为 0，应是恒等"

def _at_ex(v, m):                                 # 把单向量放到绝对位置 m
    z = torch.zeros(256, 64); z[m] = v
    return apply_rope_ex(z, cos64, sin64)[m]

qe, ke = torch.randn(64), torch.randn(64)
s1 = _at_ex(qe, 2)  @ _at_ex(ke, 5)               # 相对距离 3
s2 = _at_ex(qe, 12) @ _at_ex(ke, 15)              # 相对距离 3，绝对位置 +10
s3 = _at_ex(qe, 2)  @ _at_ex(ke, 9)               # 相对距离 7
assert torch.allclose(s1, s2, atol=1e-4), "相对位置相同 → 分数必须相同"
assert not torch.allclose(s1, s3, atol=1e-2), "相对位置不同 → 分数应不同"
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `repeat_kv`

GQA 计算时最常见的实现技巧：把 `n_kv` 个 KV 头各复制 `n_rep` 份，对齐到 `n_head` 个 query 头（transformers 库的 Llama/Qwen 源码里就有一个同名函数）。

**任务**：实现 `repeat_kv_ex(kv, n_rep)`：
- 输入 `kv` 形状 `(B, n_kv, T, d_head)`，输出 `(B, n_kv*n_rep, T, d_head)`；
- 输出第 `i` 个头必须等于输入第 `i // n_rep` 个头（同组相邻排列）；
- `n_rep == 1` 时直接返回原张量。

**提示**：插入一个新维度后 `expand` 再 `reshape`（不发生真实内存拷贝直到 reshape），约 4 行。

In [ ]:
def repeat_kv_ex(kv, n_rep):
    # kv: (B, n_kv, T, d_head) -> (B, n_kv*n_rep, T, d_head)
    # TODO: n_rep==1 时直接返回；否则 kv[:, :, None] -> expand -> reshape
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测 ----
kv0 = torch.randn(2, 4, 6, 8)
out2 = repeat_kv_ex(kv0, 3)
assert out2.shape == (2, 12, 6, 8), f"形状错误: {tuple(out2.shape)}"
for i in range(12):
    assert torch.equal(out2[:, i], kv0[:, i // 3]), f"第 {i} 个输出头应复制自第 {i//3} 个 KV 头"
assert torch.equal(repeat_kv_ex(kv0, 1), kv0), "n_rep=1 应返回原内容"
out3 = repeat_kv_ex(torch.randn(1, 1, 5, 4), 8)   # MQA 边界：1 个 KV 头供 8 个 query 头
assert out3.shape == (1, 8, 5, 4)
assert torch.equal(out3[:, 0], out3[:, 7]), "MQA：所有头应完全相同"
print("✅ 练习 2 通过")

## ✏️ 练习 3：MoE 的稀疏/稠密 FLOPs 比

把讲义 §5 的"参数-FLOPs 解耦"算成数。

**任务**：实现 `moe_flops_ratio(n_experts, top_k, d, d_ff)`，返回
**每 token 激活 top-k 个专家的 FFN FLOPs ÷ 全部 E 个专家都激活的 FLOPs**：
- 单个专家（两个矩阵 `d×d_ff` 与 `d_ff×d`）每 token 的 FLOPs 取 $2\times 2\, d\, d_{ff}$（每个乘加算 2 FLOPs）；
- 忽略 router（$2dE$，相对专家可忽略）；
- 请显式算出分子、分母再相除（虽然 $d, d_{ff}$ 会约掉——这正是要你体会的点）。

**提示**：3 行。已知点：8 个专家选 top-2 → 比值应为 $2/8 = 1/4$。

In [ ]:
def moe_flops_ratio(n_experts, top_k, d, d_ff):
    # TODO: sparse = top_k 个专家的每 token FLOPs
    # TODO: dense  = n_experts 个专家全激活的每 token FLOPs
    # TODO: 返回 sparse / dense
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
assert abs(moe_flops_ratio(8, 2, 1024, 4096) - 0.25) < 1e-12, "top-2 / 8 experts 应为 1/4"
assert abs(moe_flops_ratio(8, 1, 512, 2048) - 0.125) < 1e-12, "Switch 式 top-1 / 8 experts 应为 1/8"
assert abs(moe_flops_ratio(64, 8, 896, 4864) - 0.125) < 1e-12, "细粒度 64 选 8 也是 1/8"
assert abs(moe_flops_ratio(4, 4, 32, 64) - 1.0) < 1e-12, "全激活 = 稠密，比值 1"
r1 = moe_flops_ratio(8, 2, 1024, 4096)
r2 = moe_flops_ratio(8, 2, 4096, 14336)
assert abs(r1 - r2) < 1e-12, "比值与 d/d_ff 无关——FLOPs 只看激活了几份专家"
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。三题的实现都极短——现代架构的核心组件本来就不长，难在理解每一行为什么这么写。

In [ ]:
# ---- 练习 1 参考答案（先自己做，再对照）----
def apply_rope_ex(x, cos, sin):
    T = x.shape[-2]
    c, s = cos[:T], sin[:T]
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = torch.empty_like(x)
    out[..., 0::2] = x1 * c - x2 * s
    out[..., 1::2] = x1 * s + x2 * c
    return out
# 要点：旋转只作用于"维度对"内部，不混合不同对；cos/sin 沿最后一维对齐 d/2 对，
# 沿倒数第二维对齐位置 T，其余维度靠广播。

In [ ]:
# ---- 练习 2 参考答案（先自己做，再对照）----
def repeat_kv_ex(kv, n_rep):
    if n_rep == 1:
        return kv
    B, n_kv, T, dh = kv.shape
    return kv[:, :, None, :, :].expand(B, n_kv, n_rep, T, dh).reshape(B, n_kv * n_rep, T, dh)
# 要点：expand 不复制内存（stride=0 的虚拟维），reshape 时才物化；
# 头的排列是 [kv0,kv0,kv0, kv1,kv1,kv1, ...]，即输出头 i 对应输入头 i // n_rep。
# 生产实现（如 FlashAttention 的 GQA 内核）甚至不物化，直接在 kernel 里按组索引。

In [ ]:
# ---- 练习 3 参考答案（先自己做，再对照）----
def moe_flops_ratio(n_experts, top_k, d, d_ff):
    per_expert = 2 * 2 * d * d_ff          # 两个矩阵，每乘加 2 FLOPs
    return (top_k * per_expert) / (n_experts * per_expert)
# 要点：比值 = top_k / n_experts，与专家大小无关。
# 参数量 ∝ n_experts、计算量 ∝ top_k —— 这就是 §5 说的"解耦"：
# Mixtral-8x7B 总参 47B、激活仅 13B；按 FLOPs 算它是 13B 价钱，按知识容量算它是 47B 容量。

## 🏁 全课总结：《LLM 内核》八个模块的一条线

| 模块 | 一句话 |
|---|---|
| 00 · Setup | 全课 CPU 可跑的实验环境与课程地图 |
| 01 · Tokenization | BPE 把字节流压成 token——模型世界的第一道接口，也是很多怪 bug 的源头 |
| 02 · Attention 与 Transformer Block | QKV 内容查表 + 残差堆叠：手写出 GPT 的每一个零件 |
| 03 · 训练 mini-GPT | 把 loss 从随机水平压下去：数据、初始化、AdamW、lr schedule 每个旋钮的手感 |
| 04 · 解码策略 | 同一组 logits，greedy/temperature/top-k/top-p 解出完全不同的行为 |
| 05 · Scaling Laws | loss 是 (N, D, C) 的幂律；Chinchilla 告诉你每个 FLOP 该怎么花 |
| 06 · KV Cache 与高效推理 | decode 是显存带宽瓶颈；cache、量化、批处理都是在四本账之间倒腾 |
| 07 · 现代架构 | RoPE/GQA/RMSNorm/SwiGLU/MoE/SSM：每个改动都是参数、FLOPs、显存、质量四本账上的一笔交易 |

读任何一篇新模型的技术报告时，你现在可以直接翻到 architecture 和 config 表：每个字段在这门课里都有出处，每个改动你都能算出它买了什么、付了什么。

**评测者的最后一课**：架构决定能力剖面的形状——MoE 的知识容量、SSM 的召回短板、RoPE 缩放的检索退化，都要用针对性探针而不是单一总分去测。

🎉 **恭喜完成《LLM 内核》全部 8 个模块！** 返回 [课程主页](../index.html) 查看结课总结与后续路线。

---
## 🎯 真实数据胶囊题：真实配置上的 RoPE 旋转位置编码

RoPE 通过按维度旋转给 Q/K 注入相对位置。用真实 Pythia 的 head_dim 实现 RoPE，验证它的关键性质：旋转后 q·k 只依赖**相对**位置 (m-n)，而非绝对位置。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def load_cfg(model, url):
    p=os.path.join(CACHE,f"{model}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
PYTHIA={m:f"https://huggingface.co/EleutherAI/pythia-{m}/resolve/main/config.json"
        for m in ["160m","410m","1.4b","2.8b","6.9b","12b"]}

cfg=load_cfg("pythia-1.4b", PYTHIA["1.4b"])
d = cfg["h"]//cfg["heads"]    # 真实 head_dim
print(f"pythia-1.4b head_dim d={d}")

**练习**：实现 `rope(x, pos, d)`：对向量 `x`(长 d) 在位置 `pos` 应用 RoPE 旋转。标准做法：把维度两两配对 (2i,2i+1)，按角度 `pos·θ_i`（θ_i=10000^(-2i/d)）旋转。

In [ ]:
def rope(x, pos, d):
    # TODO: 对每对 (x[2i],x[2i+1]) 旋转角度 pos*theta_i, theta_i=10000**(-2i/d)
    raise NotImplementedError


In [ ]:
# 自测：RoPE 后内积只依赖相对位置
rng=np.random.default_rng(0); q=rng.normal(size=d); k=rng.normal(size=d)
def dot_at(m,n): return rope(q,m,d) @ rope(k,n,d)
# 相对位置相同 -> 内积相同
assert abs(dot_at(5,3) - dot_at(7,5)) < 1e-9, "RoPE: q·k 只依赖 m-n"
assert abs(dot_at(0,0) - q@k) < 1e-9, "pos=0 不旋转"
assert abs(dot_at(10,2) - dot_at(20,12)) < 1e-9
print("RoPE 相对位置性质验证通过 ✓")


### 📖 参考答案

In [ ]:
def rope(x, pos, d):
    out=x.copy().astype(float)
    for i in range(d//2):
        theta=10000**(-2*i/d); a=pos*theta; c,s=np.cos(a),np.sin(a)
        x0,x1=x[2*i],x[2*i+1]
        out[2*i]=x0*c - x1*s; out[2*i+1]=x0*s + x1*c
    return out
print("✓ RoPE 让注意力天然编码相对距离，是长上下文外推的基础")